In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import LabelEncoder, StandardScaler


## Configuration

In [12]:
INPUT_PATH = Path("output.csv")
GT_PATH = Path("Roles.csv")

ABLATION_OUTPUT_DIR = Path("outputs/ablations")
ABLATION_PLOTS_DIR = Path("plots/ablations")
INSPECTION_OUTPUT_DIR = Path("outputs/ablations/cluster_inspection")
INSPECTION_PLOTS_DIR = Path("plots/ablations/cluster_inspection")

SIDES = ["ct", "t"]
RANDOM_STATE = 69420
K_VALUES = [3, 4, 5, 6, 7, 8]
N_BOOTS = 50
RESAMPLE_FRAC = 0.80

META_COLS = {"player_name", "side", "rounds_played"}

FEATURE_GROUPS = {
    "combat": [
        "survival_rate",
        "damage_per_round",
        "damage_taken_per_round",
        "damage_diff_per_round",
        "multi_kill_rate",
        "rifle_kill_share",
        "awp_kill_share",
    ],
    "opening_aggression": [
        "opening_kill_rate",
        "opening_death_rate",
        "opening_duel_success",
        "first_contact_rate",
        "time_near_enemy_rate",
    ],
    "trading_teamwork": [
        "trade_kill_rate",
        "death_traded_rate",
        "trade_participation",
        "assists_per_round",
        "flash_assists_per_round",
    ],
    "utility": [
        "grenades_per_round",
        "he_grenades_per_round",
        "flashbangs_per_round",
        "smokes_per_round",
        "fire_nades_per_round",
        "util_damage_per_round",
    ],
    "positioning_movement": [
        "avg_distance_to_enemy",
        "avg_distance_to_team_centroid",
        "relative_team_centroid_distance",
        "avg_distance_moved_per_round",
        "avg_distance_to_closest_teammate",
        "time_stationary_rate",
    ],
}

METRIC_COLS = [
    "composite_score",
    "gt_ari",
    "gt_purity",
    "stability_mean",
    "silhouette",
]

DISPLAY_NAMES = {
    "composite_score": "Composite Score",
    "gt_ari": "Ground-Truth ARI",
    "gt_purity": "Ground-Truth Purity",
    "stability_mean": "Bootstrap Stability",
    "silhouette": "Silhouette Score",
}


## Data and ground-truth helpers

`Roles.csv` gives one role per side directly (`CT Role`, `T Role`), so
there's no dual-role majority resolution needed — a player's ground-truth
label for a side is just that column's value.

In [13]:
def load_data() -> pd.DataFrame:
    df = pd.read_csv(INPUT_PATH).fillna(0)
    required = {"player_name", "side", "rounds_played"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    df["side"] = df["side"].str.lower()
    return df[df["side"].isin(SIDES)].copy()


def load_ground_truth() -> pd.DataFrame | None:
    if not GT_PATH.exists():
        print(f"[info] no ground-truth file found at {GT_PATH}; skipping GT metrics")
        return None

    try:
        gt = pd.read_csv(GT_PATH, sep="\t", encoding="utf-16")
    except (UnicodeError, UnicodeDecodeError):
        gt = pd.read_csv(GT_PATH)

    required = {"Name", "CT Role", "T Role"}
    missing = required - set(gt.columns)
    if missing:
        print(f"[warn] {GT_PATH} is missing columns {sorted(missing)}; skipping GT metrics")
        return None

    gt = gt.fillna("")
    gt["_key"] = gt["Name"].astype(str).str.strip().str.lower()
    return gt


def add_ground_truth(df: pd.DataFrame, gt: pd.DataFrame | None, side: str) -> pd.DataFrame:
    out = df.copy()
    if gt is None:
        out["gt_role"] = ""
        return out

    role_col = "CT Role" if side == "ct" else "T Role"
    out["_key"] = out["player_name"].astype(str).str.strip().str.lower()
    out = out.merge(
        gt[["_key", role_col]].rename(columns={role_col: "gt_role"}),
        on="_key",
        how="left",
    ).drop(columns=["_key"])
    out["gt_role"] = out["gt_role"].fillna("")
    return out


def valid_role(role) -> bool:
    return str(role).strip().lower() not in {"", "unknown"}


## Feature-group ablation experiments

In [14]:
def all_feature_columns(df: pd.DataFrame) -> list[str]:
    return [
        c for c in df.columns
        if c not in META_COLS
        and not c.startswith("gt_")
        and pd.api.types.is_numeric_dtype(df[c])
        and df[c].nunique() > 1
    ]


def make_experiments(df: pd.DataFrame) -> dict[str, list[str]]:
    all_features = all_feature_columns(df)

    groups = {
        name: [c for c in cols if c in all_features]
        for name, cols in FEATURE_GROUPS.items()
    }

    std_features = [c for c in all_features if c.endswith("_std")]

    experiments = {"full": all_features}

    for group_name, cols in groups.items():
        experiments[f"{group_name}_only"] = cols
        experiments[f"no_{group_name}"] = [c for c in all_features if c not in set(cols)]

    experiments["consistency_only"] = std_features
    experiments["no_consistency"] = [c for c in all_features if c not in set(std_features)]

    weapon_cols = {"awp_kill_share", "rifle_kill_share"}
    experiments["no_weapon_share"] = [c for c in all_features if c not in weapon_cols]

    tactical_cols = (
        groups["opening_aggression"]
        + groups["trading_teamwork"]
        + groups["utility"]
        + groups["positioning_movement"]
    )
    experiments["tactics_only"] = sorted(set(tactical_cols))

    return {name: cols for name, cols in experiments.items() if len(cols) >= 2}


## Clustering, stability, and ground-truth metrics

In [15]:
def bootstrap_stability(X_scaled: np.ndarray, full_labels: np.ndarray, k: int) -> tuple[float, float]:
    rng = np.random.RandomState(RANDOM_STATE)
    n = X_scaled.shape[0]
    sample_size = max(3, int(n * RESAMPLE_FRAC))

    scores = []
    for i in range(N_BOOTS):
        idx = rng.choice(n, size=sample_size, replace=True)
        boot_labels = KMeans(
            n_clusters=k, random_state=RANDOM_STATE + i, n_init=20,
        ).fit_predict(X_scaled[idx])

        ref_labels = full_labels[idx]
        if len(set(ref_labels)) < 2 or len(set(boot_labels)) < 2:
            continue

        scores.append(adjusted_rand_score(ref_labels, boot_labels))

    if len(scores) < 5:
        return np.nan, np.nan
    return float(np.mean(scores)), float(np.std(scores))


def gt_metrics(df: pd.DataFrame, labels: np.ndarray) -> tuple[float, float]:
    """Adjusted Rand Index and cluster purity against Roles.csv, for the
    subset of players with a known, valid role on this side."""
    if "gt_role" not in df.columns:
        return np.nan, np.nan

    valid_mask = df["gt_role"].map(valid_role).to_numpy() & (labels != -1)
    if valid_mask.sum() < 2 or df.loc[valid_mask, "gt_role"].nunique() < 2:
        return np.nan, np.nan

    encoded = LabelEncoder().fit_transform(df.loc[valid_mask, "gt_role"])
    gt_ari = adjusted_rand_score(encoded, labels[valid_mask])

    temp = pd.DataFrame({
        "cluster": labels[valid_mask],
        "role": df.loc[valid_mask, "gt_role"].to_numpy(),
    })
    dominant_counts = temp.groupby("cluster")["role"].agg(lambda s: s.value_counts().max())
    gt_purity = dominant_counts.sum() / valid_mask.sum()

    return gt_ari, gt_purity


## Run the ablations

In [16]:
def run_ablation_for_side(df: pd.DataFrame, side: str, gt: pd.DataFrame | None) -> pd.DataFrame:
    side_df = df[df["side"] == side].reset_index(drop=True)
    side_df = add_ground_truth(side_df, gt, side)

    experiments = make_experiments(side_df)
    rows = []

    for ablation_name, features in experiments.items():
        X = side_df[features].apply(pd.to_numeric, errors="coerce").fillna(0)
        X = X.loc[:, X.nunique() > 1]
        if X.shape[1] < 2:
            continue

        X_scaled = StandardScaler().fit_transform(X)

        for k in K_VALUES:
            if k >= len(side_df):
                continue

            labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(X_scaled)

            sil = silhouette_score(X_scaled, labels)
            db = davies_bouldin_score(X_scaled, labels)
            stability_mean, stability_std = bootstrap_stability(X_scaled, labels, k)
            gt_ari, gt_purity = gt_metrics(side_df, labels)

            composite = (
                0.40 * sil
                + 0.20 * (1 / (1 + db))
                + 0.40 * max(0, stability_mean - stability_std)
            )

            rows.append({
                "side": side,
                "ablation": ablation_name,
                "k": k,
                "n_features": X.shape[1],
                "features": ", ".join(X.columns),
                "silhouette": sil,
                "davies_bouldin": db,
                "stability_mean": stability_mean,
                "stability_std": stability_std,
                "gt_ari": gt_ari,
                "gt_purity": gt_purity,
                "composite_score": composite,
            })

    return pd.DataFrame(rows)


def run_all_ablations() -> pd.DataFrame:
    ABLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_data()
    gt = load_ground_truth()

    all_results = []

    for side in SIDES:
        print(f"[run] ablations for {side.upper()}")
        results = run_ablation_for_side(df, side, gt)

        side_dir = ABLATION_OUTPUT_DIR / side
        side_dir.mkdir(parents=True, exist_ok=True)
        results.to_csv(side_dir / "ablation_results_all.csv", index=False)

        best = (
            results.sort_values("composite_score", ascending=False)
            .groupby("ablation", as_index=False)
            .head(1)
            .sort_values("composite_score", ascending=False)
        )
        best.to_csv(side_dir / "ablation_results_best.csv", index=False)

        print(f"\nBest ablations — {side.upper()}")
        print(best[[
            "ablation", "k", "n_features", "silhouette",
            "stability_mean", "gt_ari", "gt_purity", "composite_score",
        ]].to_string(index=False))

        all_results.append(results)

    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(ABLATION_OUTPUT_DIR / "ablation_results_all_sides.csv", index=False)

    best_combined = (
        combined.sort_values("composite_score", ascending=False)
        .groupby(["side", "ablation"], as_index=False)
        .head(1)
        .sort_values(["side", "composite_score"], ascending=[True, False])
    )
    best_combined.to_csv(ABLATION_OUTPUT_DIR / "ablation_results_best_all_sides.csv", index=False)

    print(f"\n[done] results saved in {ABLATION_OUTPUT_DIR}")
    return combined


In [ ]:
_ = run_all_ablations()


[run] ablations for CT

Best ablations — CT
                 ablation  k  n_features  silhouette  stability_mean   gt_ari  gt_purity  composite_score
    trading_teamwork_only  5           5    0.352165        0.816617 0.073946   0.489051         0.523866
              combat_only  3           7    0.313633        0.781564 0.286245   0.613139         0.468372
  opening_aggression_only  3           5    0.254112        0.831753 0.200960   0.554745         0.464919
positioning_movement_only  3           6    0.255476        0.714386 0.204932   0.576642         0.421160
             utility_only  3           6    0.227591        0.740013 0.153853   0.547445         0.398921
               no_utility  3          38    0.133701        0.783913 0.524025   0.759124         0.358505
      no_trading_teamwork  3          39    0.141025        0.765982 0.575762   0.781022         0.357190
             tactics_only  5          22    0.149044        0.686855 0.312319   0.722628         0.356666
  

## Plots

In [ ]:
def clean_ablation_name(name: str) -> str:
    return (
        name.replace("_", " ")
        .replace("positioning movement", "positioning")
        .replace("opening aggression", "opening")
        .replace("trading teamwork", "trading")
        .title()
    )


def load_best_results(side: str) -> pd.DataFrame:
    path = ABLATION_OUTPUT_DIR / side / "ablation_results_best.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run the ablations cell above first.")
    df = pd.read_csv(path)
    df["ablation_display"] = df["ablation"].apply(clean_ablation_name)
    return df


def plot_composite_bar(df: pd.DataFrame, side: str) -> None:
    work = df.dropna(subset=["composite_score"]).sort_values("composite_score", ascending=True)
    if work.empty:
        return

    fig, ax = plt.subplots(figsize=(12, max(6, len(work) * 0.45)))
    bars = ax.barh(work["ablation_display"], work["composite_score"])

    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height() / 2, f" {width:.3f}", va="center", fontsize=10)

    ax.set_xlabel("Composite Score")
    ax.set_ylabel("Ablation")
    ax.set_title(f"{side.upper()} Ablation Results — Composite Score")
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()

    out_dir = ABLATION_PLOTS_DIR / side
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{side}_composite_score_bar.png"
    plt.savefig(out, dpi=250, bbox_inches="tight")
    plt.show()
    print(f"[ok] {out}")


def plot_metric_heatmap(df: pd.DataFrame, side: str) -> None:
    available_metrics = [m for m in METRIC_COLS if m in df.columns]
    work = df[["ablation_display"] + available_metrics].copy()
    for col in available_metrics:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work.dropna(how="all", subset=available_metrics)
    if work.empty:
        return

    work = work.sort_values("composite_score", ascending=False)
    values = work[available_metrics].to_numpy(dtype=float)

    fig_h = max(6, len(work) * 0.45)
    fig_w = max(10, len(available_metrics) * 2.2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(values, aspect="auto")

    ax.set_xticks(np.arange(len(available_metrics)))
    ax.set_xticklabels([DISPLAY_NAMES.get(m, m) for m in available_metrics], rotation=30, ha="right")
    ax.set_yticks(np.arange(len(work)))
    ax.set_yticklabels(work["ablation_display"])
    ax.set_title(f"{side.upper()} Ablation Metric Heatmap")

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val = values[i, j]
            if np.isfinite(val):
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Metric Value")
    plt.tight_layout()

    out_dir = ABLATION_PLOTS_DIR / side
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / f"{side}_metric_heatmap.png"
    plt.savefig(out, dpi=250, bbox_inches="tight")
    plt.show()
    print(f"[ok] {out}")


def plot_side_comparison() -> None:
    frames = []
    for side in SIDES:
        path = ABLATION_OUTPUT_DIR / side / "ablation_results_best.csv"
        if path.exists():
            d = pd.read_csv(path)
            d["side"] = side.upper()
            d["ablation_display"] = d["ablation"].apply(clean_ablation_name)
            frames.append(d)

    if len(frames) < 2:
        return

    combined = pd.concat(frames, ignore_index=True)
    pivot = combined.pivot_table(index="ablation_display", columns="side", values="composite_score", aggfunc="mean")
    pivot = pivot.dropna()
    if pivot.empty:
        return
    if "T" in pivot.columns:
        pivot = pivot.sort_values("T", ascending=True)

    x = np.arange(len(pivot))
    width = 0.38
    fig, ax = plt.subplots(figsize=(14, max(7, len(pivot) * 0.45)))
    ax.barh(x - width / 2, pivot["CT"], height=width, label="CT")
    ax.barh(x + width / 2, pivot["T"], height=width, label="T")

    ax.set_yticks(x)
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Composite Score")
    ax.set_ylabel("Ablation")
    ax.set_title("CT vs T Ablation Comparison")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend()
    plt.tight_layout()

    ABLATION_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    out = ABLATION_PLOTS_DIR / "ct_vs_t_composite_comparison.png"
    plt.savefig(out, dpi=250, bbox_inches="tight")
    plt.show()
    print(f"[ok] {out}")


def make_all_ablation_plots() -> None:
    ABLATION_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    for side in SIDES:
        print(f"\n=== {side.upper()} Ablation Plots ===")
        df = load_best_results(side)
        plot_composite_bar(df, side)
        plot_metric_heatmap(df, side)

    plot_side_comparison()
    print(f"\n[done] ablation plots saved in {ABLATION_PLOTS_DIR}")


In [ ]:
make_all_ablation_plots()



=== CT Ablation Plots ===


FileNotFoundError: Missing outputs\ablations\ct\ablation_results_best.csv. Run the ablations cell above first.

## Selected ablation inspection

Use this section to rerun one best ablation, save player assignments, save
cluster summaries, and create a PCA inspection plot.

In [ ]:
def get_best_ablation_row(side: str, ablation: str) -> pd.Series:
    path = ABLATION_OUTPUT_DIR / side / "ablation_results_best.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run ablations first.")
    results = pd.read_csv(path)
    match = results[results["ablation"] == ablation]
    if match.empty:
        available = sorted(results["ablation"].unique())
        raise ValueError(f"Ablation '{ablation}' not found for {side}. Available: {available}")
    return match.iloc[0]


def parse_feature_list(row: pd.Series) -> list[str]:
    features = str(row["features"]).split(", ")
    return [f for f in features if f]


def build_cluster_summary(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for cluster_id, group in df.groupby("cluster"):
        row = {
            "cluster": cluster_id,
            "n_players": len(group),
            "players": ", ".join(group["player_name"].astype(str).sort_values()),
        }
        for feature in features:
            row[f"{feature}_mean"] = group[feature].mean()
        rows.append(row)
    return pd.DataFrame(rows).sort_values("cluster")


def plot_ablation_clusters(df: pd.DataFrame, side: str, ablation: str) -> None:
    plot_dir = INSPECTION_PLOTS_DIR / side
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(14, 10))
    scatter = ax.scatter(df["pc1"], df["pc2"], c=df["cluster"], s=90, cmap="tab10")
    for _, row in df.iterrows():
        ax.annotate(row["player_name"], (row["pc1"], row["pc2"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
    handles, _ = scatter.legend_elements(prop="colors")
    cluster_ids = sorted(df["cluster"].unique())
    ax.legend(handles, [f"Cluster {c}" for c in cluster_ids], title="Cluster", loc="best")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"{side.upper()} Ablated Clusters - {ablation}")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out = plot_dir / f"{ablation}_clusters_pca.png"
    plt.savefig(out, dpi=250, bbox_inches="tight")
    plt.show()
    print(f"[ok] PCA plot: {out}")


def run_selected_ablation(side: str, ablation: str) -> pd.DataFrame:
    df = load_data()
    side_df = df[df["side"] == side].reset_index(drop=True)
    row = get_best_ablation_row(side, ablation)
    k = int(row["k"])
    features = [f for f in parse_feature_list(row) if f in side_df.columns]
    if len(features) < 2:
        raise ValueError(f"Too few valid features for {side} {ablation}: {features}")

    X = side_df[features].apply(pd.to_numeric, errors="coerce").fillna(0)
    X = X.loc[:, X.nunique() > 1]
    X_scaled = StandardScaler().fit_transform(X)
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(X_scaled)
    coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)

    out = side_df.copy()
    out["cluster"] = labels
    out["pc1"] = coords[:, 0]
    out["pc2"] = coords[:, 1]
    out["ablation"] = ablation
    out["k"] = k

    side_out_dir = INSPECTION_OUTPUT_DIR / side
    side_out_dir.mkdir(parents=True, exist_ok=True)
    out_path = side_out_dir / f"{ablation}_clusters.csv"
    out.to_csv(out_path, index=False)

    summary = build_cluster_summary(out, X.columns.tolist())
    summary_path = side_out_dir / f"{ablation}_cluster_summary.csv"
    summary.to_csv(summary_path, index=False)

    plot_ablation_clusters(out, side, ablation)
    print(f"[ok] cluster assignments: {out_path}")
    print(f"[ok] cluster summary:     {summary_path}")
    return out


def print_cluster_players(df: pd.DataFrame) -> None:
    for cluster_id, group in df.groupby("cluster"):
        players = group["player_name"].astype(str).sort_values().tolist()
        print(f"\nCluster {cluster_id} | n={len(players)}")
        print(", ".join(players))


### Inspect one ablation

In [ ]:
# Example:
# selected = run_selected_ablation(side="t", ablation="no_weapon_share")
# print_cluster_players(selected)
